# Base Clients

> The core abstraction for different FL Clients. Any gneral client functionality resides here.

In [ ]:
#| default_exp clients

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fedai.clients import BaseClient
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from peft import *
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np
import math
np.random.seed(42)
torch.manual_seed(42)

<torch._C.Generator>

In [ ]:
#| export
class AgentRole(Enum):
    SERVER = 1
    CLIENT = 2
    MARL = 3

## pFedMe  
Personalized Federated learning using Morseu-envelope

In [ ]:
#| export
class pFedMe(BaseClient):
    def __init__(self, 
                 id,
                 cfg,
                 state= None,
                 role= AgentRole.CLIENT,
                 block= None):
        
        super().__init__(id, cfg, state, role, block)
        
        if self.role == AgentRole.CLIENT:
            self.local_model = deepcopy(self.model)
            self.optimizer = pFedMeOptimizer(self.model.to(self.device).parameters(), lr=self.cfg.personal_lr, lambda_=self.cfg.lambda_)
            self.label_set = list(set(np.array([batch['y'] for batch in self.train_ds])))
        

In [ ]:
#| export
@patch
def save_state(self: pFedMe, state_dict):  # noqa: F811    
    state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
    
    if not os.path.exists(os.path.dirname(state_path)):
        os.makedirs(os.path.dirname(state_path))

    self.pers_model = deepcopy(self.model).to(self.device)
    with torch.no_grad():
        for p_model, p_updated in zip(self.pers_model.parameters(), self.persionalized_model_bar):
            p_model.copy_(p_updated)

    client_state = {
        **state_dict,
        "model": self.model.state_dict(),
        'optimizer': self.optimizer.state_dict(),
        'pers_model': self.pers_model.state_dict(),
    }
    
    torch.save(client_state, state_path)

    if self.role == AgentRole.CLIENT:
        save_space(self)


In [ ]:
#| export
@patch
def communicate(self: pFedMe, another_agent: Agent):  # noqa: F811
    if self.role == AgentRole.CLIENT:
        self.save_state(self.state)

### Training

In [ ]:
#| export
@patch
def _run_batch(self: pFedMe, batch: dict) -> tuple:
    
    # find an approximate theta
    for j in range(self.cfg.K):
        self.optimizer.zero_grad()
        loss, metrics = self._closure(batch)

        if loss.item() == 0.0:
            return loss, metrics
        
        loss.backward()
        self.persionalized_model_bar, _ = self.optimizer.step(self.local_model)

    # update local weight after finding aproximate theta
    with torch.no_grad():
        for localweight, per_param in zip(self.local_model.parameters(), self.persionalized_model_bar):
            localweight.sub_(self.cfg.lambda_* self.cfg.optimizer.lr * (localweight - per_param))


    if self.cfg.model.grad_norm_clip:
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.model.grad_norm_clip)
    

    return loss, metrics

In [ ]:
#| export
@patch
def _run_epoch(self: pFedMe):

    for i, batch in enumerate(self.train_loader):
        batch = self.send_to_device(batch)
        self._run_batch(batch)

In [ ]:
#| export
@patch
def fit(self: pFedMe) -> dict:
    
    self.model = self.model.to(self.device)
    self.local_model = self.local_model.to(self.device)
    self.model.train()
    self.local_model.train()
    for _ in range(self.cfg.local_epochs):
        self._run_epoch()

    with torch.no_grad():
        for model_param, local_param in zip(self.model.parameters(), self.local_model.parameters()):
            model_param.copy_(local_param)

### Evaluation

In [ ]:
#| export
@patch
def train_test_stats(self: pFedMe, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    try:
        X, y = batch['x'], batch['y']
        logits = self.pers_model(X)
        loss = self.criterion(logits, y)
        probs = torch.nn.functional.softmax(logits, dim=-1)
        y_pred = probs.argmax(dim=-1)
        y_true = batch[self.label_key]

        if hasattr(self, "training_metrics") and self.cfg.training_metrics:
            if hasattr(self, "tokenizer"):
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true, tokenizer=self.tokenizer)
            else:
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true)

    except Exception as e:
        print(f"Error in _closure Eval_test_stats: {e}")
        return torch.tensor(0.0, dtype=torch.float32, device=self.device), metrics  # Return safe values

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: pFedMe, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    # self.per_model = deepcopy(self.model)
    # self.per_model = self.per_model.to(self.device)

    # with torch.no_grad():
    #     for param, per_param in zip(self.per_model.parameters(), self.persionalized_model_bar.to(self.device).parameters()):
    #         param.copy_(per_param)
    
    self.pers_model.eval()
    self.pers_model = self.pers_model.to(self.device)

    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    return {"loss": avg_loss, "metrics": total_metrics}


The server initiate the evaluation process, which computes the local metrics and average them.

In [ ]:
#| export
@patch
def evaluate(self: pFedMe, t):
    self.cfg.agg ="mtl"
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):

        client = self.client_fn(self.client_cls, self.cfg, id, self.latest_round, t, self.loss_fn, state_dir= self.cfg.state_dir)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
    self.cfg.agg = "one_model"
    return lst_train_res, lst_test_res    


### Aggregation

In [ ]:
#| export
@patch
def aggregate(self: pFedMe, lst_active_ids, comm_round, len_clients_ds):

    m_t = sum(len_clients_ds.values())
    with torch.no_grad():

        if comm_round > 1:
            prev_server_state_path = os.path.join(self.cfg.save_dir, str(comm_round - 1), "global_model", "state.pth")
            prev_server_state = torch.load(prev_server_state_path, weights_only=False)
            prev_global_model = prev_server_state['model']

        else:
            id = lst_active_ids[0]
            prev_server_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
            prev_server_state = torch.load(prev_server_state_path, weights_only=False)
            prev_global_model = prev_server_state['w0'].to(self.device).state_dict()


        for i, id in enumerate(lst_active_ids):
            state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
            state = torch.load(state_path, weights_only=False)
            client_state_dict = state['model']

            if i == 0:
                global_model = {
                    key: torch.zeros_like(value) 
                    for key, value in client_state_dict.items()
                }

            n_k = len_clients_ds[id]
            weight =  n_k / m_t 

            for key in client_state_dict.keys():
                global_model[key].add_(weight * client_state_dict[key])

        for key in global_model.keys():
            global_model[key].copy_((1-self.cfg.beta)*global_model[key] + self.cfg.beta*prev_global_model[key])

        server_state = {
            'model': global_model,
        }

        server_state_path = os.path.join(self.cfg.save_dir, str(comm_round), "global_model", "state.pth")
        
        if not os.path.exists(os.path.dirname(server_state_path)):
            os.makedirs(os.path.dirname(server_state_path), exist_ok=True)
            
        torch.save(server_state, server_state_path)

    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()